У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [185]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks
from collections import Counter

In [186]:
# Load data
df = pd.read_csv('customer_segmentation_train.csv')

df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [187]:
display(df.info())
print('\n')
display('Has missing values: ', df.isnull().values.any())
print('\n')
display('Columns with missing values: ', df.columns[df.isnull().any()])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


None

'Has missing values: '

np.True_

'Columns with missing values: '

Index(['Ever_Married', 'Graduated', 'Profession', 'Work_Experience',
       'Family_Size', 'Var_1'],
      dtype='object')

In [188]:
null_count = df.isnull().sum()
null_percentage = round((df.isnull().sum()/df.shape[0])*100, 2)

null_df = pd.DataFrame({'column_name' : df.columns,'null_count' : null_count,'null_percentage': null_percentage})
null_df.reset_index(drop = True, inplace = True)
null_df.sort_values(by = 'null_percentage', ascending = False)

,column_name,null_count,null_percentage
6,Work_Experience,829,10.28
8,Family_Size,335,4.15
2,Ever_Married,140,1.74
5,Profession,124,1.54
4,Graduated,78,0.97
9,Var_1,76,0.94
0,ID,0,0.00
1,Gender,0,0.00
3,Age,0,0.00
7,Spending_Score,0,0.00


## `Work_Experience` imputation (829 rows, 10%):

In [189]:
display(df['Work_Experience'].value_counts())
print('\n')
print('Work_Experience mode: ', df['Work_Experience'].mode()[0])
print('Work_Experience median: ', df['Work_Experience'].median())

,count
Work_Experience,
1.0,2354
0.0,2318
9.0,474
8.0,463
2.0,286
3.0,255
4.0,253
6.0,204
7.0,196




Work_Experience mode:  1.0
Work_Experience median:  1.0


In [190]:
#df['Work_Experience_imputed'] = df['Work_Experience'].isnull()
mode_work_experience = df['Work_Experience'].mode()[0]
df['Work_Experience'] = df['Work_Experience'].fillna(mode_work_experience)

## `Family_Size` imputation (335 rows, 4%)


In [191]:
display(df['Family_Size'].value_counts())
print('\n')
print('Family_Size mode: ', df['Family_Size'].mode()[0])
print('Family_Size median: ', df['Family_Size'].median())

,count
Family_Size,
2.0,2390
3.0,1497
1.0,1453
4.0,1379
5.0,612
6.0,212
7.0,96
8.0,50
9.0,44




Family_Size mode:  2.0
Family_Size median:  3.0


In [192]:
#df['Family_Size_imputed'] = df['Family_Size'].isnull()
mode_family_size = df['Family_Size'].mode()[0]
df['Family_Size'] = df['Family_Size'].fillna(mode_family_size)

## `Ever_Married` imputing (140 rows, 1.7%)

In [193]:
display(df[df['Ever_Married'].isnull()])

display(df['Ever_Married'].value_counts())

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
24,461021,Female,NaN,58,No,Executive,1.0,Average,3.0,Cat_3,B
108,466466,Female,NaN,19,No,Healthcare,6.0,Low,5.0,Cat_3,D
201,466065,Male,NaN,19,No,Healthcare,9.0,Low,3.0,Cat_3,D
213,460516,Female,NaN,85,No,Lawyer,0.0,High,1.0,Cat_3,C
272,464841,Male,NaN,19,No,Entertainment,0.0,High,3.0,Cat_4,D
...,...,...,...,...,...,...,...,...,...,...,...
7756,465987,Male,NaN,20,No,Healthcare,1.0,Low,3.0,Cat_2,D
7775,462989,Male,NaN,32,Yes,Healthcare,1.0,Low,1.0,Cat_6,D
8011,466026,Female,NaN,49,No,Entertainment,0.0,Low,1.0,Cat_3,A
8030,459082,Male,NaN,45,Yes,Artist,1.0,Low,2.0,Cat_6,A


,count
Ever_Married,
Yes,4643
No,3285


In [194]:
df['Ever_Married'] = df['Ever_Married'].fillna('NA') # or we could transform column into 1/0 column and impute with mode value

## `Profession` imputing (124, 1.5%)

In [195]:
display(df[df['Profession'].isnull()])

display(df['Profession'].value_counts())

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
79,466567,Female,No,49,Yes,NaN,1.0,Low,1.0,Cat_6,D
118,465827,Male,No,27,No,NaN,2.0,Low,7.0,Cat_4,D
219,465837,Male,No,62,Yes,NaN,0.0,Low,1.0,Cat_6,A
237,467252,Female,No,33,Yes,NaN,0.0,Low,4.0,NaN,D
437,461410,Male,Yes,79,No,NaN,0.0,Average,2.0,NaN,C
...,...,...,...,...,...,...,...,...,...,...,...
7743,467388,Male,No,35,NaN,NaN,0.0,Low,2.0,Cat_6,D
7870,462301,Female,No,27,No,NaN,12.0,Low,3.0,Cat_6,D
7899,464548,Female,Yes,47,No,NaN,1.0,Low,1.0,Cat_4,A
7935,464977,Female,Yes,66,No,NaN,1.0,Average,2.0,Cat_4,B


,count
Profession,
Artist,2516
Healthcare,1332
Entertainment,949
Engineer,699
Doctor,688
Lawyer,623
Executive,599
Marketing,292
Homemaker,246


In [196]:
df['Profession'] = df['Profession'].fillna('NA')

## `Graduated` imputation (78rows, 1%)

In [197]:
display(df[df['Graduated'].isnull()])

display(df['Graduated'].value_counts())

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
57,462267,Male,No,32,NaN,Doctor,8.0,Low,2.0,Cat_6,D
220,464613,Female,No,35,NaN,Artist,0.0,Low,3.0,Cat_6,B
290,465058,Female,No,43,NaN,Entertainment,1.0,Low,1.0,Cat_6,B
431,462548,Male,No,18,NaN,Executive,1.0,Low,5.0,Cat_4,A
510,460685,Male,No,51,NaN,Artist,6.0,Low,4.0,Cat_4,B
...,...,...,...,...,...,...,...,...,...,...,...
7602,466506,Male,Yes,50,NaN,Artist,1.0,Average,4.0,Cat_6,A
7743,467388,Male,No,35,NaN,NA,0.0,Low,2.0,Cat_6,D
7829,460567,Male,No,39,NaN,Entertainment,8.0,Low,4.0,Cat_3,C
7987,462933,Male,Yes,55,NaN,Entertainment,1.0,High,5.0,Cat_6,B


,count
Graduated,
Yes,4968
No,3022


In [198]:
df['Graduated'] = df['Graduated'].fillna('NA') # or we could transform column into 1/0 column and impute with mode value

## `Var_1` imputation (76rows, 1%)

In [199]:
display(df[df['Var_1'].isnull()])

display(df['Var_1'].value_counts())

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
43,466006,Female,Yes,49,Yes,Artist,1.0,Low,2.0,NaN,C
163,463595,Female,No,32,No,Healthcare,1.0,Low,5.0,NaN,C
186,459576,Female,Yes,85,No,Lawyer,1.0,Low,2.0,NaN,A
231,463467,Female,No,23,No,Healthcare,0.0,Low,4.0,NaN,D
233,461282,Male,No,21,No,Healthcare,0.0,Low,4.0,NaN,D
...,...,...,...,...,...,...,...,...,...,...,...
7548,466477,Male,Yes,36,Yes,Artist,1.0,Average,2.0,NaN,C
7715,459724,Female,Yes,39,No,Artist,0.0,Average,4.0,NaN,A
7761,462935,Male,Yes,79,No,Entertainment,1.0,Low,1.0,NaN,D
7913,467544,Male,Yes,56,No,Artist,0.0,Low,2.0,NaN,C


,count
Var_1,
Cat_6,5238
Cat_4,1089
Cat_3,822
Cat_2,422
Cat_7,203
Cat_1,133
Cat_5,85


In [200]:
df['Var_1'] = df['Var_1'].fillna('NA')

In [201]:
display('Nulls count: ', df.isnull().sum())

'Nulls count: '

,0
ID,0
Gender,0
Ever_Married,0
Age,0
Graduated,0
Profession,0
Work_Experience,0
Spending_Score,0
Family_Size,0
Var_1,0


## Preprocessing

In [202]:
# Get train and test datasets:
input_cols = list(df.columns)[1:-1]
target_col = 'Segmentation'
X = df[input_cols].copy()
y = df[target_col].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

scaler = MinMaxScaler()
scaler.fit(X_train[numeric_cols])
X_train[numeric_cols] = scaler.transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough' # Keep numerical columns as they are
)

# Fit the preprocessor on X_train and transform X_train
X_train_processed_data = preprocessor.fit_transform(X_train)
X_test_processed_data = preprocessor.transform(X_test)

# Get feature names after one-hot encoding for the transformed X_train
ohe_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
# Numerical columns are passed through and will be at the end of the transformed array
all_transformed_cols = list(ohe_feature_names) + numeric_cols

# Convert the processed array back to a DataFrame for easier handling
X_train_processed = pd.DataFrame(X_train_processed_data, columns=all_transformed_cols, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed_data, columns=all_transformed_cols, index=X_test.index)

In [203]:
df[target_col].value_counts()

,count
Segmentation,
D,2268
A,1972
C,1970
B,1858


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [204]:
# Get indices of categorical columns for SMOTENC
categorical_features_indices = [X_train.columns.get_loc(col) for col in categorical_cols]

print(f"Numerical columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")
print(f"Categorical feature indices: {categorical_features_indices}")

print('\n')

Numerical columns: ['Age', 'Work_Experience', 'Family_Size']
Categorical columns: ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']
Categorical feature indices: [0, 1, 3, 4, 6, 8]




In [205]:
print('SMOTENC')
print(f"Original class distribution: {Counter(y_train)}")

smotenc = SMOTENC(categorical_features=categorical_features_indices, random_state=42)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, y_train)

X_train_smotenc_processed_data = preprocessor.transform(X_train_smotenc)
X_train_smotenc_processed = pd.DataFrame(X_train_smotenc_processed_data, columns=all_transformed_cols, index=X_train_smotenc.index)

print(f"SMOTENC resampled class distribution: {Counter(y_train_smotenc)}")

SMOTENC
Original class distribution: Counter({'D': 1814, 'A': 1578, 'C': 1576, 'B': 1486})
SMOTENC resampled class distribution: Counter({'A': 1814, 'B': 1814, 'C': 1814, 'D': 1814})


In [206]:
print('SMOTE-Tomek')
print(f"Original class distribution: {Counter(y_train)}")

# We already have SmoteNC, now create TomekLinks:
tomek = TomekLinks()

# Combine SMOTENC with TomekLinks:
smt = SMOTETomek(smote=smotenc, tomek=tomek)
# this did not work, couldn't figure out why I keep getting: `ValueError: could not convert string to float: 'Female'`
# X_res_smotetomek, y_res_smotetomek = smt.fit_resample(X_train, y_train)

# Since X_train_processed is now all numerical, use standard SMOTE for the oversampling part
smotetomek = SMOTETomek(smote=SMOTE(random_state=42), random_state=42)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train_processed, y_train)

print(f"SMOTE-Tomek resampled class distribution: {Counter(y_train_smotetomek)}")

SMOTE-Tomek
Original class distribution: Counter({'D': 1814, 'A': 1578, 'C': 1576, 'B': 1486})
SMOTE-Tomek resampled class distribution: Counter({'C': 1471, 'D': 1454, 'B': 1400, 'A': 1373})


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [207]:
# 1. one-vs-rest (OvR) on processed train data:
print('one-vs-rest (OvR) on processed train data: \n')
log_reg = LogisticRegression(solver='liblinear')
ovr_model_basic = OneVsRestClassifier(log_reg)
ovr_model_basic.fit(X_train_processed, y_train)
ovr_predictions_basic = ovr_model_basic.predict(X_test_processed)

# Обчислимо метрики precision та recall для кожного класу
print(classification_report(y_test, ovr_predictions_basic))

one-vs-rest (OvR) on processed train data: 

              precision    recall  f1-score   support

           A       0.42      0.46      0.44       394
           B       0.42      0.17      0.24       372
           C       0.49      0.63      0.55       394
           D       0.65      0.76      0.70       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.48      1614
weighted avg       0.50      0.52      0.50      1614



In [208]:
# 2. one-vs-rest (OvR) on SMOTENC data:
print('one-vs-rest (OvR) on SMOTENC train data: \n')
log_reg_smotenc = LogisticRegression(solver='liblinear')
ovr_model_smotenc = OneVsRestClassifier(log_reg_smotenc)
ovr_model_smotenc.fit(X_train_smotenc_processed, y_train_smotenc)
ovr_predictions_smotenc = ovr_model_smotenc.predict(X_test_processed)

# Обчислимо метрики precision та recall для кожного класу
print(classification_report(y_test, ovr_predictions_smotenc))

one-vs-rest (OvR) on SMOTENC train data: 

              precision    recall  f1-score   support

           A       0.42      0.47      0.44       394
           B       0.40      0.25      0.31       372
           C       0.52      0.59      0.55       394
           D       0.67      0.72      0.70       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.50      1614
weighted avg       0.51      0.52      0.51      1614



In [209]:
# 3. one-vs-rest (OvR) on SMOTE-Tomek data:
print('one-vs-rest (OvR) on SMOTE-Tomek train data: \n')
log_reg_smotetomek = LogisticRegression(solver='liblinear')
ovr_model_smotetomek = OneVsRestClassifier(log_reg_smotetomek)
ovr_model_smotetomek.fit(X_train_smotetomek, y_train_smotetomek)
ovr_predictions_smotetomek = ovr_model_smotetomek.predict(X_test_processed)

# Обчислимо метрики precision та recall для кожного класу
print(classification_report(y_test, ovr_predictions_smotetomek))

one-vs-rest (OvR) on SMOTE-Tomek train data: 

              precision    recall  f1-score   support

           A       0.43      0.49      0.46       394
           B       0.39      0.24      0.30       372
           C       0.50      0.60      0.54       394
           D       0.68      0.70      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.50      1614
weighted avg       0.51      0.52      0.51      1614



In [210]:
# Усереднені метрики macro та micro для OvR
ovr_macro_precision_basic = precision_score(y_test, ovr_predictions_basic, average='macro')
ovr_micro_precision_basic = precision_score(y_test, ovr_predictions_basic, average='micro')
ovr_macro_recall_basic = recall_score(y_test, ovr_predictions_basic, average='macro')
ovr_micro_recall_basic = recall_score(y_test, ovr_predictions_basic, average='micro')

# Усереднені метрики macro та micro для OvR on SMOTENC
ovo_macro_precision_smotenc = precision_score(y_test, ovr_predictions_smotenc, average='macro')
ovo_micro_precision_smotenc = precision_score(y_test, ovr_predictions_smotenc, average='micro')
ovo_macro_recall_smotenc = recall_score(y_test, ovr_predictions_smotenc, average='macro')
ovo_micro_recall_smotenc = recall_score(y_test, ovr_predictions_smotenc, average='micro')


# Усереднені метрики macro та micro для OvR on SMOTE-Tomek
ovo_macro_precision_smotetomek = precision_score(y_test, ovr_predictions_smotetomek, average='macro')
ovo_micro_precision_smotetomek = precision_score(y_test, ovr_predictions_smotetomek, average='micro')
ovo_macro_recall_smotetomek = recall_score(y_test, ovr_predictions_smotetomek, average='macro')
ovo_micro_recall_smotetomek = recall_score(y_test, ovr_predictions_smotetomek, average='micro')

# Створимо датафрейм для відображення результатів
results = pd.DataFrame({
    'Metric': ['Macro Precision', 'Micro Precision', 'Macro Recall', 'Micro Recall'],
    'OvR': [ovr_macro_precision_basic, ovr_micro_precision_basic, ovr_macro_recall_basic, ovr_micro_recall_basic],
    'OvR (SMOTE)': [ovo_macro_precision_smotenc, ovo_micro_precision_smotenc, ovo_macro_recall_smotenc, ovo_micro_recall_smotenc],
    'OvR (SMOTE-Tomek)': [ovo_macro_precision_smotetomek, ovo_micro_precision_smotetomek, ovo_macro_recall_smotetomek, ovo_micro_recall_smotetomek]
})
print(results)

            Metric       OvR  OvR (SMOTE)  OvR (SMOTE-Tomek)
0  Macro Precision  0.496299     0.500841           0.500231
1  Micro Precision  0.519827     0.519207           0.517348
2     Macro Recall  0.505626     0.507711           0.506412
3     Micro Recall  0.519827     0.519207           0.517348


### Спостереження

1.  **Порівняння за метрикою Micro Precision**:

    *   **OvR (Original Data)**: Micro Precision = 0.5198
    *   **OvR (SMOTENC)**: Micro Precision = 0.5192
    *   **OvR (SMOTE-Tomek)**: Micro Precision = 0.5173

2. **Найкраща модель**:

    Модель, натренована на **оригінальних даних (OvR)**, демонструє незначно вищу Micro Precision (0.5198) порівняно з моделями, натренованими на даних, збалансованих за допомогою SMOTENC (0.5192) та SMOTE-Tomek (0.5173).

3.  **Гіпотеза щодо відсутності суттєвої різниці**:

    Різниця в показниках Micro Precision між усіма трьома моделями є дуже незначною, що свідчить про відсутність суттєвого покращення продуктивності після застосування методів ресемплінгу (SMOTE та SMOTE-Tomek). Можливо причина в тому, що:

    *   **Недостатньо сильний дисбаланс класів**: Можливо, початковий дисбаланс класів у датасеті був не настільки критичним, щоб суттєво вплинути на загальну точність класифікатора логістичної регресії:
    
        Segmentation
        - D	2268
        - A	1972
        - C	1970
        - B	1858

    *   **Обмеження моделі**: Логістична регресія є лінійною моделлю. Можливо, для більш складних взаємодій у даних, які могли б бути краще вивчені на збалансованих даних, потрібна більш складна модель класифікації, яка б ефективніше використовувала згенеровані SMOTE зразки.

    У цьому випадку, хоча SMOTENC і SMOTE-Tomek успішно збалансували класи, це не призвело до відчутного покращення Micro Precision на тестових даних, що може свідчити про те, що для цієї конкретної задачі та моделі, ресемплінг не є ключовим фактором підвищення продуктивності.